### Moving beyond a single agent architecture

this will take a couple of weeks as the tools we will be implementing will need their own implementation 

these will be Speaicalist agents

1. shopping cart agents 
2. warehouse management agent 
3. QnA agent (already built)

we will have these three agants, eventually implementing a coordinator/supervisor to manage all of these agents

Along with evals for multi agent systems 

and moving onto a to a communication 

going to also move the tools into a util script (out of the jupiter notebooks)

### Shopping cart agent & tools

this will - look up items, add items, delete items from shoppign carts

all of this will be stores in postgress db

To explore the database without a terminal we wil use dbeaver 

`langgraph_db:5433`
`langgraph_user`
`langgraph_password`

 now we can build out the database for our tools i.e shopping cart etc

in the root please see new folder `scripts/sql/tools_database.sql`

In [ ]:
CREATE USER tools_user WITH PASSWORD 'tools_user_password';
CREATE DATABASE tools_database OWNER tools_user;
GRANT ALL PRIVILEGES ON DATABASE tools_database TO tools_user;

in prod you would create a set of tables and set the permissions to ONLY the neceassary tables not whole db as we have here 

now we create the shopping cart table

In [ ]:
-- create new schema for shopping cart tools
-- in the real world youd have sepereate databases for each tools (especially if each owned by a different team)
-- in this case we are seperating the tools within the db with individual schemas
CREATE SCHEMA IF NOT EXISTS shopping_carts; 

CREATE TABLE shopping_carts.shopping_cart_items (
    id SERIAL PRIMARY KEY,
    user_id VARCHAR(255) NOT NULL,
    shopping_cart_id VARCHAR(255) NOT NULL DEFAULT 'main',
    product_id VARCHAR(255) NOT NULL,
    -- not productionr ready - irl we would have a dimensional table 
    -- info such as price, image url, etc would live in their own tables and be joined using the product id
    -- lengthy data will be duplicated with current implementation
    price DECIMAL(10, 2),
    quantity INTEGER NOT NULL DEFAULT 1,
    currency VARCHAR(3) DEFAULT 'USD',
    product_image_url VARCHAR(1000),
    added_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    
    -- Constraints
    CONSTRAINT positive_price CHECK (price >= 0),
    CONSTRAINT positive_quantity CHECK (quantity > 0),
    CONSTRAINT unique_user_cart_product UNIQUE (user_id, shopping_cart_id, product_id)
);

-- Index for faster queries by user and cart
CREATE INDEX idx_shopping_cart_user_cart ON shopping_carts.shopping_cart_items(user_id, shopping_cart_id);

-- Index for faster queries by user
CREATE INDEX idx_shopping_cart_user_id ON shopping_carts.shopping_cart_items(user_id);

-- Index for faster queries by product
CREATE INDEX idx_shopping_cart_product_id ON shopping_carts.shopping_cart_items(product_id);

-- Trigger to automatically update the updated_at timestamp
-- programatically trigger the below any time anything is done to this table
CREATE OR REPLACE FUNCTION update_shopping_cart_timestamp()
RETURNS TRIGGER AS $$
BEGIN
    NEW.updated_at = CURRENT_TIMESTAMP;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER shopping_cart_update_timestamp
    BEFORE UPDATE ON shopping_carts.shopping_cart_items
    FOR EACH ROW
    EXECUTE FUNCTION update_shopping_cart_timestamp();

run the above by connecting to the previously created tools database 

`tools_database`
`tools_user`
`tools_user_password`

now we will build three tools that allow us to 
1. add items 
2. get items
3. delete items

### Shopping cart agent tools

### Import Dependencies

In [ ]:
from pydantic import BaseModel

import cohere

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langsmith import traceable, get_current_run_tree

import instructor

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send

from langchain_core.messages import SystemMessage, convert_to_openai_messages, HumanMessage, AIMessage
from IPython.display import Image, display

from typing import Literal, Dict, Any, Annotated, List
from pydantic import Field
from operator import add

import random
import openai
import pandas as pd

from jinja2 import Template

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Document, Prefetch, FusionQuery, FieldCondition, MatchAny, Filter, MatchValue

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np

`uv add --dev psycopg2`

### Add to Shopping Cart Tool
A list of items 
each represented w/ product id (from data in qdrant)

In [ ]:
items = [
    {
        "product_id": "B0BLTPRG3X",
        "quantity": 2
    },
    {
        "product_id": "B08NYL2Z72",
        "quantity": 4
    }
]

the tool below takes the list above, the user and cart_id

with description for agent 

In [ ]:
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:

    """Add a list of provided items to the shopping cart.
    
    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.
        
    Returns:
        A list of the items added to the shopping cart.
    """

### connect to the database
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

### this context manager helps dealing with the outputs of the queries 
### the results will be transformed into lists of dictionaries automatically
    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
        for item in items:
            ### set id and qty
            product_id = item['product_id']
            quantity = item['quantity']

            ### init qdrant db to retrieve additional product info
            qdrant_client = QdrantClient(url="http://localhost:6333")

            ### query data - retrieve single items
            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id)
                                )
                            ]
                    ),
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            ).points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = 'USD'
        
            # Check if item already exists in cart
            ### the query below is properly implemented 
            ### using f string would not suffice 
            ### this will implement a number of under the hood checks on the query 
            check_query = """
                SELECT id, quantity, price 
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()
            
            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity
                
                ### update any data if it changes for items already in the cart i.e price
                update_query = """
                    UPDATE shopping_carts.shopping_cart_items 
                    SET 
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """
                
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                
                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))
            
    return f"Added {items} to the shopping cart."
    ### should add a try accept to this so it fails gracefully (currently it doesnt)

In [ ]:
add_to_shopping_cart(items, "test_user_1", "test_cart_1")

In [ ]:
items_2 = [
    {
        "product_id": "B08NYL2Z72",
        "quantity": 4
    }
]

In [ ]:
add_to_shopping_cart(items_2, "test_user_1", "test_cart_1")

In [ ]:
items_3 = [
    {
        "product_id": "B08NYL2Z72",
        "quantity": 3
    },
    {
        "product_id": "B09WR36NK8",
        "quantity": 1
    },
]

In [ ]:
add_to_shopping_cart(items_3, "test_user_1", "test_cart_1")

### Get the Shopping Cart Items Tool

In [ ]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:

    """
    Retrieve all items in a user's shopping cart.
    
    Args:
        user_id: User identifier
        cart_id: Cart identifier
    
    Returns:
        List of dictionaries containing cart items
    """
    ### connect to db
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                SELECT 
                    product_id, price, quantity,
                    currency, product_image_url,
                    (price * quantity) as total_price
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s
                ORDER BY added_at DESC
            """
        ### execute query
        cursor.execute(query, (user_id, cart_id))

        ### transform and return
        return [dict(row) for row in cursor.fetchall()]

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")

In [ ]:
get_shopping_cart("sdgsfgdfss", "hgklfhjlhjlhgj")

### Deleting items from the Shopping Cart Tool

(this only allows completeley removing items from basket - regardless of number in cart i.e cant do partial deletes)

In [ ]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:

    """
    Remove an item completely from the shopping cart.
    
    Args:
        user_id: User identifier
        product_id: Product identifier to remove
        cart_id: Cart identifier
    
    Returns:
        Information about the removal of the item from the shopping cart.
    """
    
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                DELETE FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
        cursor.execute(query, (user_id, cart_id, product_id))

        return f"Removed {product_id} from the shopping cart." if cursor.rowcount > 0 else f"Item {product_id} not found in the shopping cart."

In [ ]:
remove_from_cart("B08NYL2Z72", "test_user_1", "test_cart_1")

In [ ]:
remove_from_cart("B08NYL2Z72", "test_user_1", "test_cart_1")

In [ ]:
add_to_shopping_cart(items_3, "test_user_1", "test_cart_1")

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")

In [ ]:
remove_from_cart("B0BLTPRG3X", "test_user_1", "test_cart_1")

In [ ]:
get_shopping_cart("test_user_1", "test_cart_1")